# Milestone 6 - Part 2: Multi-Tool Agent

This notebook runs the Part 2 agent workflow with two tools:
- `retrieve_wiki` (retriever from Part 1)
- `summarize_text` (additional tool)

At each step, the model reads mission + history and outputs one JSON action:
`thought`, `tool`, `arguments`, then the controller executes it until `finish`.

Submission requirements covered here:
- local open-weight 7B-14B instruct model (`mistral:7b-instruct` via Ollama)
- observable traces in `agent_traces/`
- 10 multi-step tasks from `data/agent_tasks.json`


## Step 0: Environment Check

Open this notebook from the repository root so `ROOT` resolves correctly.

Required packages are in `requirements.txt`.

If the notebook does not run, the kernel likely failed to start. Re-select kernel, reload the IDE window, or run the notebook from terminal Jupyter.


In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT =", ROOT)


## Step 1: Build Shared Retrieval Pipeline

This uses `rag_core.RAGPipeline` (same as Part 1):
- load corpus
- chunk text
- build embeddings
- create FAISS index

First run may take longer due to model loading and embedding.


In [ ]:
from rag_core import RAGPipeline

pipeline = RAGPipeline(ROOT)
print("chunks:", len(pipeline.chunks), "index:", pipeline.index.ntotal)


## Step 2: Single Task Demo

Run one task first and inspect trace structure (`thought`, `tool`, `arguments`, observations).


In [ ]:
from agent_runner import run_episode, load_tasks

TASKS_PATH = ROOT / "data" / "agent_tasks.json"
tasks = load_tasks(TASKS_PATH)
demo = tasks[0]
print("Demo task:", demo["id"], demo["title"])

trace = run_episode(demo["mission"], pipeline, model="mistral:7b-instruct", max_steps=8)
trace["task_id"] = demo["id"]
print(json.dumps(trace, ensure_ascii=False, indent=2)[:4500])


## Step 3: Generate 10 Traces

Run all tasks and write:
`agent_traces/t01_trace.json` to `agent_traces/t10_trace.json`.

This can take several minutes depending on local model speed.


In [ ]:
from agent_runner import run_all_tasks

run_all_tasks(ROOT, TASKS_PATH, ROOT / "agent_traces", model="mistral:7b-instruct")


## Step 4: Final Deliverables for Part 2

Before submission:
1. confirm all 10 traces exist in `agent_traces/`
2. ensure `agent_report.md` summarizes policy, performance, failures, and latency
3. verify README includes setup, usage, architecture, limitations, and model serving command
